<a href="https://colab.research.google.com/github/narmathakathiravan98/gdg-hackathon-2026/blob/main/GoogleHackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-huggingface



In [ ]:
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage
from google.colab import userdata

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HF_TOKEN")

model = ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation"
))

# ── Wrap as call_llm so rest of pipeline works ──
def call_llm(prompt: str) -> str:
    result = model.invoke([HumanMessage(content=prompt)])
    return result.content

# Test
test = call_llm('Return ONLY this JSON: {"status": "ok"}')
print(f"✅ LLM working: {test.strip()[:50]}")

✅ LLM working: {"status": "ok"}


In [ ]:
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from sentence_transformers import SentenceTransformer
import chromadb, re, os
from typing import List
from google.colab import userdata

In [ ]:
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage
from google.colab import userdata

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HF_TOKEN")

model = ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation"
))

# ── Wrap as call_llm so rest of pipeline works ──
def call_llm(prompt: str) -> str:
    result = model.invoke([HumanMessage(content=prompt)])
    return result.content

# Test
test = call_llm('Return ONLY this JSON: {"status": "ok"}')
print(f"✅ LLM working: {test.strip()[:50]}")

✅ LLM working: {"status": "ok"}


In [ ]:
KEYWORD_CATEGORIES = {
    "bulk_delete": {
        "keywords": ["delete", "invalidSet"],
        "prompt": """You are an HDFS expert. This log line shows a block deletion event.
A block is being added to invalidSet meaning HDFS marked it for deletion.

Log line:
{log_line}

Return ONLY this JSON:
{{
  "category": "bulk_delete",
  "service_name": "dfs.FSNamesystem",
  "timestamp": "extract from log",
  "affected_node": "extract IP",
  "error_severity": "HIGH",
  "likely_cause": "stale replica OR job cleanup OR over-replication",
  "suggested_remediation": "verify replication factor, check if node is being decommissioned"
}}
JSON:"""
    },

    "transfer_failure": {
        "keywords": ["Got exception", "DataXceiver"],
        "prompt": """You are an HDFS expert. This log shows a DataNode block transfer failure.

Log line:
{log_line}

Return ONLY this JSON:
{{
  "category": "transfer_failure",
  "service_name": "dfs.DataNode$DataXceiver",
  "timestamp": "extract from log",
  "source_node": "extract source IP",
  "destination_node": "extract destination IP",
  "error_severity": "HIGH",
  "likely_cause": "network timeout OR connection reset OR node overload",
  "suggested_remediation": "check network between nodes, restart DataNode if repeated"
}}
JSON:"""
    },

    "block_replication": {
        "keywords": ["addStoredBlock", "blockMap updated"],
        "prompt": """You are an HDFS expert. This log shows a block replication event.

Log line:
{log_line}

Return ONLY this JSON:
{{
  "category": "block_replication",
  "service_name": "dfs.FSNamesystem",
  "timestamp": "extract from log",
  "affected_node": "extract IP",
  "block_size": "extract size if present",
  "error_severity": "LOW",
  "likely_cause": "normal replication OR re-replication after failure",
  "suggested_remediation": "monitor replication factor, no action if within normal range"
}}
JSON:"""
    },

    "partial_block": {
        "keywords": ["allocateBlock"],
        "prompt": """You are an HDFS expert. This log shows a new block being allocated.
Partial blocks smaller than 64MB indicate end-of-file or interrupted writes.

Log line:
{log_line}

Return ONLY this JSON:
{{
  "category": "partial_block",
  "service_name": "dfs.FSNamesystem",
  "timestamp": "extract from log",
  "file_path": "extract path from log",
  "error_severity": "MEDIUM",
  "likely_cause": "end of file write OR job completing OR interrupted write",
  "suggested_remediation": "verify job completed successfully, check for incomplete writes"
}}
JSON:"""
    },

    "block_verification": {
        "keywords": ["Verification succeeded", "DataBlockScanner"],
        "prompt": """You are an HDFS expert. This log shows a block integrity scan result.

Log line:
{log_line}

Return ONLY this JSON:
{{
  "category": "block_verification",
  "service_name": "dfs.DataBlockScanner",
  "timestamp": "extract from log",
  "block_id": "extract block ID",
  "error_severity": "LOW",
  "likely_cause": "routine integrity check",
  "suggested_remediation": "no action needed, block is healthy"
}}
JSON:"""
    },

    "concurrent_jobs": {
        "keywords": ["task_", "allocateBlock"],
        "prompt": """You are an HDFS expert. This log shows a MapReduce task writing blocks.

Log line:
{log_line}

Return ONLY this JSON:
{{
  "category": "concurrent_jobs",
  "service_name": "dfs.FSNamesystem",
  "timestamp": "extract from log",
  "job_id": "extract task ID",
  "file_path": "extract path",
  "error_severity": "LOW",
  "likely_cause": "normal job execution OR concurrent job contention",
  "suggested_remediation": "monitor job queue, reduce concurrency if DataNodes are overloaded"
}}
JSON:"""
    }
}

print(f"✅ {len(KEYWORD_CATEGORIES)} categories defined")

✅ 6 categories defined


In [ ]:
from typing import Dict
def categorize_lines(filepath: str) -> Dict[str, List[str]]:
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        all_lines = [l.strip() for l in f if l.strip()]

    categorized = {cat: [] for cat in KEYWORD_CATEGORIES}
    uncategorized = []

    for line in all_lines:
        matched = False
        for cat, config in KEYWORD_CATEGORIES.items():
            if any(kw in line for kw in config["keywords"]):
                categorized[cat].append(line)
                matched = True
                break
        if not matched:
            uncategorized.append(line)

    # Print summary
    print(f"📂 Total lines: {len(all_lines)}\n")
    for cat, lines in categorized.items():
        print(f"  {cat:25s} → {len(lines):4d} lines")
    print(f"  {'uncategorized':25s} → {len(uncategorized):4d} lines")

    return categorized

categorized = categorize_lines("HDFS_2k.log")

📂 Total lines: 2000

  bulk_delete               →  229 lines
  transfer_failure          →  454 lines
  block_replication         →  314 lines
  partial_block             →  115 lines
  block_verification        →   20 lines
  concurrent_jobs           →    0 lines
  uncategorized             →  868 lines


In [ ]:
all_results = {}

for category, lines in categorized.items():
    if not lines:
        continue

    config  = KEYWORD_CATEGORIES[category]
    results = []
    print(f"\n📦 [{category}] — {len(lines)} lines")

    for i, line in enumerate(lines):
        try:
            filled_prompt = config["prompt"].format(log_line=line)
            text          = call_llm(filled_prompt)   # ← Gemini
            data          = extract_json(text)
            if data:
                results.append(data)
                print(f"  {i+1}/{len(lines)} ✅ [{data.get('error_severity','?')}] {data.get('likely_cause','')[:55]}")
            else:
                print(f"  {i+1}/{len(lines)} ⚠️ skipped")
        except Exception as e:
            print(f"  {i+1}/{len(lines)} ❌ ({e})")

    all_results[category] = results

# Save
output = {
    cat: {"total": len(categorized[cat]), "extracted": len(res), "events": res}
    for cat, res in all_results.items()
}
with open("hdfs_analysis.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"\n✅ Saved to hdfs_analysis.json")
print(json.dumps({k: v["extracted"] for k, v in output.items()}, indent=2))


📦 [bulk_delete] — 229 lines
  1/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  2/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  3/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  4/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  5/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  6/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  7/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  8/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  9/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  10/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  11/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  12/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  13/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  14/229 ✅ [HIGH] stale replica OR job cleanup OR over-replication
  15/229 ✅ [HIGH] stale replica OR job cle

KeyboardInterrupt: 